In [1]:
import requests


In [2]:
BASE_URL = "https://api.mangadex.org"
title = "a"

# Search manga title

In [ ]:
def get_manga_with_japanese_panels(title=""):
    # Step 1: Broad search for titles matching the query
    search_params = {
        "limit": 10,
        "title": title,
        "availableTranslatedLanguage[]": ["ja"], # Metadata says JP exists
        "hasAvailableChapters": "true",
        "contentRating[]": ["safe", "suggestive"],
    }
    
    r = requests.get(f"{BASE_URL}/manga", params=search_params)
    results = r.json().get("data", [])

    candidates = []
    for m in results:
        m_id = m["id"]
        
        # Step 2: The "Internal Truth" Test
        # We need to make sure the Japanese chapters aren't just external links
        feed_params = {
            "translatedLanguage[]": ["ja"],
            "includeExternalUrl": 0, # REQUIRE hosted images for your proxy
            "includeEmptyPages": 0,
            "limit": 1,
        }
        r_feed = requests.get(f"{BASE_URL}/manga/{m_id}/feed", params=feed_params)
        
        # If total > 0, MangaDex is hosting actual Japanese images for this title
        if r_feed.json().get("total", 0) > 0:
            candidates.append(m)
            if len(candidates) >= 10: break
            
    return candidates

In [ ]:
def getCoverArt(manga):
    filename = ""
    for rel in manga.get("relationships", []):
        if rel["type"] == "cover_art":
            filename = rel["attributes"]["fileName"]
    url = f"https://uploads.mangadex.org/covers/{manga['id']}/{filename}.256.jpg"
    return url

def getCover(manga_id, filename):
    url = f"https://uploads.mangadex.org/covers/{manga_id}/{filename}.256.jpg"
    return url

In [39]:
jp_mangas = get_manga_with_japanese_panels("")
len(jp_mangas)

2

In [41]:
filename = getCoverArt(jp_mangas[0])
filename

'https://uploads.mangadex.org/covers/e9b61bac-799f-4da2-83c1-cedb2f5a3778/2830d8a1-d272-42b2-983a-9db5c7680217.jpg.256.jpg'

In [253]:
search_params = {
    "title": title,
    "limit": 20,
    "originalLanguage[]": ["ja"],
    "hasAvailableChapters": "true",
    # "availableTranslatedLanguage[]": ["en"],
}
r_search = requests.get(f"{BASE_URL}/manga", params=search_params)
    
manga = r_search.json().get("data", [])
print(len(manga))

20


In [280]:
manga_id = jp_mangas[0]['id']
manga_id

'4162b06b-3f7e-4e6e-b27b-1e48b88c5204'

# Get chapters

In [281]:
feed_params = {
        "order[chapter]": "desc",
        "translatedLanguage[]": ["ja"],
        "limit": 5,
        "includeEmptyPages": 0,
        "includeExternalUrl": 0
    }
r_feed = requests.get(f"{BASE_URL}/manga/{manga_id}/feed", params=feed_params)
chapters = r_feed.json().get("data", [])

In [282]:
chapter_id = chapters[0]['id'] #first chapter (latest chapter)'s id
url = chapters[0]['attributes']['externalUrl']
chapters[0]

{'id': 'f6aa71cd-dca7-4692-a780-16af09ba9220',
 'type': 'chapter',
 'attributes': {'volume': '1',
  'chapter': '5',
  'title': '復帰初日',
  'translatedLanguage': 'ja',
  'externalUrl': None,
  'isUnavailable': False,
  'publishAt': '2025-09-24T10:20:31+00:00',
  'readableAt': '2025-09-24T10:20:31+00:00',
  'createdAt': '2025-09-24T10:20:30+00:00',
  'updatedAt': '2025-09-24T10:20:59+00:00',
  'version': 3,
  'pages': 17},
 'relationships': [{'id': '9e1917be-0bb6-4d5e-bdf3-23a437b000cc',
   'type': 'scanlation_group'},
  {'id': '4162b06b-3f7e-4e6e-b27b-1e48b88c5204', 'type': 'manga'},
  {'id': '68324452-6a30-47f5-825a-3730d154e546', 'type': 'user'}]}

In [190]:
!mloader {url} -r

Usage: mloader [OPTIONS] [URLS]...
Try 'mloader --help' for help.

Error: Invalid value for '[URLS]...': Invalid url: None


# NOTE: If pages = 0 or theres an externalURL, it means the manga is not hosted on mangadex so we can't download the images!

In [283]:
def get_chapter_panels(id):
    # 1. Ask MangaDex which server to use
    r = requests.get(f"https://api.mangadex.org/at-home/server/{id}")
    data = r.json()
    if data['result'] == 'error':
        print("No chapter, probably hosted externally.")
        return []
    # 2. Grab the base URL and the chapter-specific hash
    base_url = data["baseUrl"]
    chapter_hash = data["chapter"]["hash"]
    
    # 'data' contains the high-quality filenames
    # 'dataSaver' contains the compressed/smaller filenames
    file_names = data["chapter"]["data"] 
    
    # 3. Construct the full URL for every page
    # Format: {baseUrl}/data/{hash}/{filename}
    urls = [f"{base_url}/data/{chapter_hash}/{name}" for name in file_names]
    
    return urls

In [284]:
panel_urls = get_chapter_panels(chapter_id)

if panel_urls:
    first_panel = panel_urls[0]
    first_panel

In [285]:
panel_urls[:3]

['https://cmdxd98sb0x3yprd.mangadex.network/data/6b628c086c336340d0060ef63d79dc53/1-4049c6fbe8f3fe8d848147ae77615adcccbe95ca4b77a93d90b0af1e37ee726b.jpg',
 'https://cmdxd98sb0x3yprd.mangadex.network/data/6b628c086c336340d0060ef63d79dc53/2-ad191f5195d0ea395915ce3fc169010b46026770d13768e0e2fc00b8db3667cb.jpg',
 'https://cmdxd98sb0x3yprd.mangadex.network/data/6b628c086c336340d0060ef63d79dc53/3-e19ef1f538c8a29217dce10a68552012039a2ad418dea8f8ae27df5512db955a.jpg']

# downloading the chapter locally

In [12]:
def download_panel(url, filename="manga_panel.jpg"):
    # 1. Send a GET request to the image URL
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        # 2. Open a local file in 'wb' (write binary) mode
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"Success! Saved as {filename}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")

In [14]:
test = "https://cmdxd98sb0x3yprd.mangadex.network/data-saver/4007d56744b3ae0b1dd54707fc4d780a/1-dc301bd32e67ad8e11113b4c2aa5626a2117c4463152408f62c0e7ad81a13bce.jpg"
download_panel(test, "test.jpg")

Success! Saved as test.jpg


# testing mloader to download frm mangaplus

In [36]:
url = chapters[0]['attributes']['externalUrl']
url

'https://mangaplus.shueisha.co.jp/viewer/1028362'

In [37]:
!mloader {url} -r


           _                 _
 _ __ ___ | | ___   __ _  __| | ___ _ __
| '_ ` _ \| |/ _ \ / _` |/ _` |/ _ \ '__|
| | | | | | | (_) | (_| | (_| |  __/ |
|_| |_| |_|_|\___/ \__,_|\__,_|\___|_|

13.03.2026 14:18:01 |   INFO   |  __main__.py   206  | Started export
13.03.2026 14:18:03 |   INFO   |   loader.py    140  | 1/1) Manga: One Piece
13.03.2026 14:18:03 |   INFO   |   loader.py    141  |     Author: Eiichiro Oda
13.03.2026 14:18:03 |   INFO   |   loader.py    152  |     1/1) Chapter #1176: Chapter 1176: With Pride
#1176
13.03.2026 14:18:26 |   INFO   |  __main__.py   227  | SUCCESS
